# coarse

## Import

In [1]:
import os
import gc
import math
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
from glob import glob
from sklearn.model_selection import KFold

sys.path.append("../input/pythonbox")
from box import Box

# Torch
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.io import read_image
from torch.utils.data import DataLoader, Dataset 
from timm import create_model
from torchvision.models import efficientnet_v2_l, EfficientNet_V2_L_Weights

# Lightning
import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule, LightningModule, seed_everything
from pytorch_lightning.callbacks import ModelCheckpoint, TQDMProgressBar, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

print(pl.__version__)
warnings.filterwarnings("ignore")

## Config

In [2]:
config = {'exp_name':'baseline_v1',
          'seed': 2023,
          'root': '../input/petfinder-pawpularity-score/', 
          'n_splits': 5,
          'n_epochs': 10,
          'early_stop': 5,
          'lr': 1e-4,
          'pretrain': True,
          'image_size': 400,
          'train_loader': {
              'batch_size': 16,
              'shuffle': True,
              'num_workers': os.cpu_count(),
              'pin_memory': True,
              'drop_last': False
          },
          'val_loader': {
              'batch_size': 16,
              'shuffle': False,
              'num_workers': os.cpu_count(),
              'pin_memory': True,
              'drop_last': False
          },
          'test_loader': {
              'batch_size': 16,
              'shuffle': False,
              'num_workers': os.cpu_count(),
              'pin_memory': True,
              'drop_last': False
          },
          'model':{
              'name': 'efficientnetv2_l',
              'output_dim': 1
          },
          'loss': 'nn.BCEWithLogitsLoss',
}

config = Box(config)

## Fix Seed

In [3]:
seed_everything(config.seed)

Global seed set to 2023


2023

## Tools

In [4]:
def mixup(x: torch.Tensor, y: torch.Tensor, alpha: float = 1.0):
    assert alpha > 0, "alpha should be larger than 0"
    assert x.size(0) > 1, "Mixup cannot be applied to a single instance."

    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(x.size()[0])
    mixed_x = lam * x + (1 - lam) * x[rand_index, :]
    target_a, target_b = y, y[rand_index]
    return mixed_x, target_a, target_b, lam


def rmse(predict,target):
    return 100. * torch.sqrt(nn.MSELoss()(predict, target))

## Dataset

In [5]:
class PetfinderDataset(Dataset):
    """Dataset
    Args:
        df: the dataframe from csv, and the "Id" column needs to be the path of Image
    """
    def __init__(self, df, transform=None, image_size=224):
        
        self._X = df["Id"].values
        self._y = None
        self.transform = transform
        
        # 判斷有沒有分數
        if "Pawpularity" in df.keys():
            self._y = df["Pawpularity"].values
            
    def __len__(self):
        return len(self._X)

    def __getitem__(self, idx):
        image_path = self._X[idx]
        image = read_image(image_path)
        image = self.transform(image)
        
        if self._y is not None:
            label = self._y[idx]
            return image, label
        return image

## Model

In [14]:
class Model(pl.LightningModule):
    def __init__(self, hparams):
        super().__init__()
        self.save_hyperparameters(hparams)  # 儲存超參數
        
        self._criterion = eval(self.hparams.loss)()
        self.metrics = rmse
        
        self.validation_step_outputs = {'val/logits': [],
                                        'val/pred': [],
                                        'val/labels': []}  # 用來計算epoch的val/loss, val/rmse
        
        self.__build_model()
        
    def __build_model(self):
        # self.backbone = create_model(self.hparams.model.name,
        #                              pretrained=self.hparams.pretrain, 
        #                              num_classes=0, 
        #                              in_chans=3)
        # num_features = self.backbone.num_features
        
        self.backbone = efficientnet_v2_l(weights=EfficientNet_V2_L_Weights.DEFAULT)
        num_features = 1000
        self.fc = nn.Sequential(nn.Dropout(0.5), 
                                nn.Linear(num_features, 
                                          self.hparams.model.output_dim))

    def forward(self, x):
        f = self.backbone(x)
        out = self.fc(f)
        return out

    def training_step(self, batch, batch_idx):
        images, labels = batch
        
        # Mixup (50%的機率)
        if torch.rand(1)[0] < 0.5:
            mix_images, target_a, target_b, lam = mixup(images, labels, alpha=0.5)
            logits = self(mix_images).squeeze()
            loss = self._criterion(logits, target_a) * lam + (1 - lam) * self._criterion(logits, target_b)
        else:
            logits = self(images).squeeze()
            loss = self._criterion(logits, labels)
            
        self.log("train/loss", loss, prog_bar=True)
        return loss
        
    def validation_step(self, batch, batch_idx):
        images, labels = batch
        
        logits = self(images).squeeze()
        loss = self._criterion(logits, labels)
        
        self.validation_step_outputs['val/logits'].append(logits)
        self.validation_step_outputs['val/labels'].append(labels)
        
        return {'val/loss': loss}
    
    def on_validation_epoch_end(self):
        logits = torch.cat(self.validation_step_outputs['val/logits'], dim=0)
        labels = torch.cat(self.validation_step_outputs['val/labels'], dim=0)
        pred = torch.sigmoid(logits)
        
        loss = self._criterion(logits, labels)
        metric = self.metrics(pred, labels)
        
        self.log('val/loss', loss, prog_bar=True)
        self.log('val/metric', metric, prog_bar=True)
        
        self.validation_step_outputs['val/logits'].clear()
        self.validation_step_outputs['val/pred'].clear()
        self.validation_step_outputs['val/labels'].clear()

    def predict_step(self, batch, batch_idx):
        images = batch
        pred = self(images).squeeze().sigmoid()  # return後會自動轉成numpy
        return pred
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        return optimizer

## Test

In [32]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]  # RGB
IMAGENET_STD = [0.229, 0.224, 0.225]  # RGB

test_transform = T.Compose([T.Resize([config.image_size, config.image_size]),
                            # T.CenterCrop([config.image_size, config.image_size]),
                            T.ConvertImageDtype(torch.float),
                            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])

stage = 'train'
df = pd.read_csv(os.path.join(config.root, stage+'.csv'))
df["Id"] = df["Id"].apply(lambda x: os.path.join(config.root, stage, x + ".jpg")) # 將ID改成圖片路徑
if "Pawpularity" in df.keys():
    test_y = df["Pawpularity"].astype(float).apply(lambda x: x / 100.).to_numpy() # 將Pawpularity軟換到[0, 1]
    df.drop('Pawpularity', axis=1, inplace=True)

# df to dataset
predict_data = PetfinderDataset(df, test_transform, config.image_size)
predict_loader = DataLoader(predict_data, **config.test_loader)

### kFold predictions

In [17]:

warnings.filterwarnings("ignore")  # 關閉 Warning
torch.set_float32_matmul_precision('high')  # 設置高精度(根據顯卡調整)

total_predictions = []
for fold in range(config.n_splits):
    model_weight = glob(f'../input/efficientnet_v2/fold_{fold}/version_0/*.ckpt')[0]

    model = Model(config).load_from_checkpoint(model_weight)

    trainer = pl.Trainer(logger=False)
    predictions = trainer.predict(model, dataloaders=predict_loader)
    
    total_predictions.append(np.concatenate(predictions).flatten())
    
    # 清除變數與快取
    del model, trainer, predictions
    torch.cuda.empty_cache()
    gc.collect()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: 0it [00:00, ?it/s]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: 0it [00:00, ?it/s]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: 0it [00:00, ?it/s]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: 0it [00:00, ?it/s]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: 0it [00:00, ?it/s]

### 計算平均預測分數

In [27]:
mean_predictions = np.array(total_predictions).mean(axis=0)
print(mean_predictions)

[0.376326   0.4389245  0.3966761  ... 0.24187501 0.37074688 0.4052097 ]


### 計算RMSE

In [40]:
rmse(torch.tensor(mean_predictions), torch.tensor(test_y)).item()

16.228203830960037

### 儲存平均預測分數

In [26]:
# 儲存csv
stage = 'train'
df = pd.read_csv(os.path.join(config.root, stage+'.csv'))
df_id = df['Id']

mean_predicts_df = pd.DataFrame(mean_predictions, columns=["Pawpularity"])
df = pd.concat([df_id, mean_predicts_df], axis=1)
df.to_csv("submission.csv", index=False)
df.head(5)

,Id,Pawpularity
0,0007de18844b0dbbb5e1f607da0606e0,0.376326
1,0009c66b9439883ba2750fb825e1d7db,0.438924
2,0013fd999caf9a3efe1352ca1b0d937e,0.396676
3,0018df346ac9c1d8413cfcc888ca8246,0.492508
4,001dc955e10590d3ca4673f034feeef2,0.426755
